# Phase 6 - Conservative LoD1 buildings

This notebook creates defensible flat-roof LoD1 blocks for the complete UQ pilot footprint inventory. It uses Queensland generated outlines, Phase 5 class-6 LiDAR and the one-metre DTM, while retaining withheld and suspicious cases instead of inventing geometry.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / 'pyproject.toml').exists():
    project_root = project_root.parent
if not (project_root / 'pyproject.toml').exists():
    raise RuntimeError('Could not locate the project root containing pyproject.toml')

src_path = str(project_root / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import geopandas as gpd
import pandas as pd
from IPython.display import JSON, Image, display

from uqgems.phase6 import run_phase6

project_root

## Run and validate

The initial run scans the analysis LAZ in chunks. A later run reuses processed products only when source, configuration and output hashes match the manifest.

In [ ]:
summary = run_phase6(project_root)
print(f"Phase 6 status: {summary['status']}")
print(f"Cached model products reused: {summary['cache_reused']}")
display(JSON(summary['checks'], expanded=True))

## Pilot result

A withheld height means the source outline remains in the audit dataset but no unsupported solid is added to the presentation mesh.

In [ ]:
model = summary['processing_statistics']['building_model']
assignment = summary['processing_statistics']['point_assignment']
overview = pd.Series({
    'primary footprint records': model['footprints'],
    'LoD1 blocks generated': model['modelled'],
    'heights withheld': model['withheld'],
    'specifically named outlines': model['named'],
    'boundary-clipped outlines': model['boundary_clipped'],
    'median modelled height (m)': model['height_m']['median'],
    'median roof-point density (points/m2)': model['roof_support']['median_density_points_m2'],
    'median roof-cell coverage': model['roof_support']['median_cell_coverage'],
    'class-6 points assigned to footprints': (
        assignment['class6_points_assigned_to_primary_footprints']
    ),
    'watertight GLB meshes': summary['presentation_mesh']['watertight_geometry_count'],
}, name='value')
display(overview.to_frame())
display(pd.Series(model['modelled_confidence_counts'], name='modelled buildings').to_frame())

## Building-level evidence

The GeoPackage is authoritative. The table below exposes the height calculation and source-support fields used for review.

In [ ]:
building_path = project_root / summary['outputs']['gpkg']
buildings = gpd.read_file(building_path, layer='lod1_buildings')
columns = [
    'building_id', 'building_name', 'model_status', 'height_m', 'confidence',
    'roof_points_used', 'roof_point_density_m2', 'roof_cell_coverage',
    'roof_z_nmad_m', 'bsm_edge_difference_m', 'issues',
]
display(buildings[columns].sort_values('building_id'))

In [ ]:
print('Withheld outlines')
display(buildings.loc[buildings['model_status'] != 'modelled', columns])
review_path = project_root / summary['outputs']['tables']['mismatch']
print('Potentially incomplete or misaligned public-source cases')
display(pd.read_csv(review_path))

## Fixed visual checks

The oblique figures use three-times vertical exaggeration for inspection only. The GeoPackage and GLB retain true elevations and scale.

In [ ]:
figure_order = [
    'plan', 'roof_points', 'isolated_roofs', 'height_distribution',
    'northeast', 'southwest',
]
for name in figure_order:
    print(name.replace('_', ' ').title())
    display(Image(filename=str(project_root / summary['outputs']['figures'][name])))

## Phase 7 handoff

The GeoPackage is the source of truth for selection and analysis; the GLB is the presentation asset. LoD2 feasibility should begin with non-boundary buildings that have strong roof-cell coverage and low roof dispersion. Read `PHASE6_LOD1_BUILDINGS.md` before interpreting low-confidence or suspicious cases.